# ZA-GAS model comparison: phi-only vs phi-xi

This notebook runs the comparison requested for the thesis workflow:

- `phi` only time-varying, with `xi` static.
- `phi` and `xi` both time-varying.
- In-sample PIT fitting and quantile-residual ACF.
- 95% confidence intervals for estimated hyper-parameters.
- Kupiec and Christoffersen 95% coverage tests for IS, OOS, and full sample.
- IS/OOS CRPS, log likelihood, AIC, BIC, RMSE, and MAD.
- OOS expected value and quantiles: 50%, 75%, 90%, 95%, 97%, 99%, simulated min, simulated max.
- CSV cache files for costly results, plus PDF and LaTeX report output.

The CSV and `.tex` cache outputs are ignored by git. The PDF is intentionally not globally ignored.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from models.factory import build_zagas_model
from models.parameters import load_theta_csv
from diagnostics.residuals import quantile_residuals, pit_values
from diagnostics.tests import jarque_bera
from report.workflow import evaluate_and_cache_model
from report.summarize import display_model_label, render_model_report

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
print('Imports ok')

Imports ok


## Configuration

Use `RUN_SCOPE` to decide what is estimated or loaded in this notebook:

- `RUN_SCOPE = 'monthly'`: run/load only the monthly models. Monthly estimation uses the thesis trial specification: standard `BFGS` with `use_bounds=False`, saved under `MONTHLY_BFGS_CACHE_DIR`.
- `RUN_SCOPE = 'daily'`: run/load only the daily models, using the existing daily cache and leaving monthly results out of the report.
- `RUN_SCOPE = 'both'`: load daily models from cache, run/load monthly models, and create the combined report.

Set `N_MAX_LOCATIONS = None` to run every train/test pair. During development, keep it at `1` so you do not accidentally launch all costly fits.

Current data folders:

- Daily: `C:/Users/ilang/OneDrive/Documentos/Ilan/academia/dissertação/data/output`
- Monthly: `C:/Users/ilang/OneDrive/Documentos/Ilan/academia/dissertação/data/monthly output`

To regenerate the report without re-estimating, keep `FORCE_REFIT = False`, keep the same cache directories, and rerun the executable cells from **Configuration** through **Render PDF and LaTeX report**. Cached `estimated_parameters.csv` files are loaded instead of optimizing again. Daily refits are blocked unless you explicitly set `ALLOW_DAILY_REFIT = True`. When you change `MONTHLY_DIR` to a different dataset, also use a new `MONTHLY_BFGS_CACHE_DIR` or set `FORCE_REFIT = True` intentionally.

In [2]:
DAILY_DIR = Path(r'C:/Users/ilang/OneDrive/Documentos/Ilan/academia/dissertação/data/output')
MONTHLY_DIR = Path(r'C:/Users/ilang/OneDrive/Documentos/Ilan/academia/dissertação/data/monthly output')

Y_COL = 'PRECIPITACAO TOTAL, DIARIO(mm)'
DATE_COL = 'Data Medicao'

N_MAX_LOCATIONS = 1
RUN_SCOPE = 'daily'  # choose: 'monthly', 'daily', or 'both'
if RUN_SCOPE not in {'monthly', 'daily', 'both'}:
    raise ValueError("RUN_SCOPE must be one of: 'monthly', 'daily', 'both'")
RUN_DAILY = RUN_SCOPE in {'daily', 'both'}
RUN_MONTHLY = RUN_SCOPE in {'monthly', 'both'}

MODEL_TYPES = ['phi', 'phi_xi']
FORCE_REFIT = True
N_DRAWS = 500
SEED = 42

DAILY_LAG_TAG = 'lags_1_365_366_050626'
CACHE_DIR = Path(f'artifacts/cache_daily_{DAILY_LAG_TAG}')
REPORT_DIR = Path(f'artifacts/reports_daily_{DAILY_LAG_TAG}')
MONTHLY_BFGS_CACHE_DIR = Path('artifacts/cache_monthly_output_bfgs_216')
ALLOW_DAILY_REFIT = True
MONTHLY_FIT_METHOD = 'BFGS'
MONTHLY_USE_BOUNDS = False
USE_MONTHLY_BFGS_TRIAL = True
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MONTHLY_BFGS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def load_pairs(directory: Path, y_col: str, date_col: str, n_max=None) -> dict:
    train_files = sorted(f for f in directory.iterdir() if f.name.endswith('_train.csv'))
    if n_max is not None:
        train_files = train_files[:n_max]

    out = {}
    for train_file in train_files:
        loc = train_file.stem.replace('_train', '')
        test_file = train_file.parent / f'{loc}_test.csv'
        if not test_file.exists():
            print(f'WARNING: no test file for {loc}; skipping')
            continue

        df_train = pd.read_csv(train_file)
        df_test = pd.read_csv(test_file)
        out[loc] = {
            'y_train': df_train[y_col].to_numpy(dtype=float),
            'y_test': df_test[y_col].to_numpy(dtype=float),
            'date_train': df_train[date_col].to_numpy() if date_col in df_train.columns else None,
            'date_test': df_test[date_col].to_numpy() if date_col in df_test.columns else None,
            'train_file': train_file,
            'test_file': test_file,
        }
    return out

daily_data = load_pairs(DAILY_DIR, Y_COL, DATE_COL, N_MAX_LOCATIONS) if RUN_DAILY else {}
monthly_data = load_pairs(MONTHLY_DIR, Y_COL, DATE_COL, N_MAX_LOCATIONS) if RUN_MONTHLY else {}

print('Daily locations:', list(daily_data))
print('Monthly locations:', list(monthly_data))

Daily locations: ['BELO HORIZONTE']
Monthly locations: []


## Parameter-count sanity check

For daily `phi_xi`, the expected total is 43: two time-varying positive-part parameters, seven GAS lags, two static GB2 parameters, and nine pi-dynamics parameters.

In [4]:
for seasonal in [s for s in ['monthly', 'daily'] if (s == 'monthly' and RUN_MONTHLY) or (s == 'daily' and RUN_DAILY)]:
    for model_type in MODEL_TYPES:
        model = build_zagas_model(model_type, seasonal=seasonal)
        print(model_type, seasonal, model.parameter_count_breakdown())

phi daily {'seasonal': 'daily', 'lags': [1, 365, 366], 'n_lags': 3, 'tv_parameters': ['phi'], 'per_tv_parameter': 8, 'gas_dynamic_parameters': 8, 'static_positive_parameters': 3, 'pi_parameters': 5, 'total_parameters': 16, 'formula': 'n_tv * (omega + f0 + A_lags + B_lags) + n_static_positive + n_pi'}
phi_xi daily {'seasonal': 'daily', 'lags': [1, 365, 366], 'n_lags': 3, 'tv_parameters': ['phi', 'xi'], 'per_tv_parameter': 8, 'gas_dynamic_parameters': 16, 'static_positive_parameters': 2, 'pi_parameters': 5, 'total_parameters': 23, 'formula': 'n_tv * (omega + f0 + A_lags + B_lags) + n_static_positive + n_pi'}


## Fit or load models, then cache diagnostics

If an `estimated_parameters.csv` already exists and `FORCE_REFIT = False`, the notebook reloads `theta` from CSV instead of re-estimating. If the rest of the cached diagnostic CSVs exist, those are loaded too. Otherwise, diagnostics are recomputed from the loaded parameters.

For monthly models, this notebook uses the BFGS/no-bounds specification configured above:

```python
MONTHLY_FIT_METHOD = 'BFGS'
MONTHLY_USE_BOUNDS = False
MONTHLY_BFGS_CACHE_DIR = Path('artifacts/cache_monthly_output_bfgs_216')
```

To run only monthly models on the larger monthly dataset, set `RUN_SCOPE = 'monthly'`, confirm `MONTHLY_DIR` points to the desired folder, and run the notebook from **Configuration** through this section. To keep the daily models in the same report while updating only monthly results, set `RUN_SCOPE = 'both'`; daily will load from cache when `FORCE_REFIT = False`. To regenerate the report from cached results, keep `FORCE_REFIT = False` and rerun the cells through **Render PDF and LaTeX report**.

In [5]:
def safe_model_id(seasonal: str, loc: str, model_type: str) -> str:
    clean_loc = ''.join(ch if ch.isalnum() else '_' for ch in loc).strip('_').lower()
    return f'{seasonal}_{clean_loc}_{model_type}'


def series_id(model_id: str) -> str:
    if model_id.endswith('_phi_xi'):
        return model_id[:-7]
    if model_id.endswith('_phi'):
        return model_id[:-4]
    return model_id


def make_series_summary(model_id: str, data: dict) -> pd.DataFrame:
    rows = []
    for sample, values in [
        ('IS', np.asarray(data['y_train'], dtype=float)),
        ('OOS', np.asarray(data['y_test'], dtype=float)),
        ('Full', np.concatenate([np.asarray(data['y_train'], dtype=float), np.asarray(data['y_test'], dtype=float)])),
    ]:
        x = values[np.isfinite(values)]
        qs = np.quantile(x, [0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
        rows.append({
            'series_id': series_id(model_id),
            'model_id': model_id,
            'sample': sample,
            'n': len(x),
            'min': np.min(x),
            'max': np.max(x),
            'mean': np.mean(x),
            'std': np.std(x, ddof=1),
            'prop_zero': np.mean(x == 0.0),
            'q01': qs[0], 'q05': qs[1], 'q25': qs[2], 'q50': qs[3],
            'q75': qs[4], 'q90': qs[5], 'q95': qs[6], 'q99': qs[7],
        })
    return pd.DataFrame(rows)


def make_series_values(model_id: str, data: dict) -> pd.DataFrame:
    return pd.concat([
        pd.DataFrame({
            'series_id': series_id(model_id),
            'model_id': model_id,
            'sample': 'IS',
            't_index': np.arange(len(data['y_train'])),
            'y': np.asarray(data['y_train'], dtype=float),
        }),
        pd.DataFrame({
            'series_id': series_id(model_id),
            'model_id': model_id,
            'sample': 'OOS',
            't_index': np.arange(len(data['y_test'])),
            'y': np.asarray(data['y_test'], dtype=float),
        }),
    ], ignore_index=True)


def make_is_diagnostic_series(model, theta, model_id: str, data: dict) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    paths = model.filter(theta, data['y_train'])
    cdfs = model.cdf_series(theta, data['y_train'])
    y_eff = paths['y_eff']
    is_pit = pit_values(cdfs, y_eff, randomise_zeros=True, rng=rng)
    is_qr = quantile_residuals(cdfs, y_eff, randomise_zeros=True, rng=rng)
    return pd.DataFrame({
        'model_id': model_id,
        'sample': 'IS',
        't_index': np.arange(len(is_pit)),
        'pit': is_pit,
        'quantile_residual': is_qr,
    })


def make_jb_from_diagnostic_series(model_id: str, diagnostic_series: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for sample in ['IS']:
        qr = diagnostic_series.loc[
            (diagnostic_series['model_id'] == model_id) & (diagnostic_series['sample'] == sample),
            'quantile_residual'
        ].dropna().to_numpy()
        d = jarque_bera(qr)
        rows.append({
            'model_id': model_id,
            'sample': sample,
            'n': d['n'],
            'skewness': d['skewness'],
            'kurtosis': d['kurtosis'],
            'JB_stat': d['stat'],
            'pvalue': d['pvalue'],
            'reject_5pct': d['pvalue'] < 0.05,
        })
    return pd.DataFrame(rows)


def cache_base_for_model(seasonal: str, model_id: str) -> Path:
    if seasonal == 'monthly' and USE_MONTHLY_BFGS_TRIAL:
        return MONTHLY_BFGS_CACHE_DIR / model_id
    return CACHE_DIR / model_id


def cache_output_dir_for_seasonal(seasonal: str) -> Path:
    if seasonal == 'monthly' and USE_MONTHLY_BFGS_TRIAL:
        return MONTHLY_BFGS_CACHE_DIR
    return CACHE_DIR


def cached_result(model, model_id: str, data: dict):
    seasonal = 'monthly' if model_id.startswith('monthly_') else 'daily'
    base = cache_base_for_model(seasonal, model_id)
    required = {
        'parameters': base / 'estimated_parameters.csv',
        'metrics': base / 'metrics.csv',
        'oos_forecasts': base / 'oos_forecasts.csv',
        'acf': base / 'quantile_residual_acf.csv',
        'pit': base / 'pit_fit.csv',
        'coverage': base / 'coverage_95.csv',
    }
    if not all(path.exists() for path in required.values()):
        return None

    result = {
        'model_id': model_id,
        'model': model,
        'parameters': pd.read_csv(required['parameters']),
        'metrics_frame': pd.read_csv(required['metrics']),
        'oos_forecasts': pd.read_csv(required['oos_forecasts']),
        'acf': pd.read_csv(required['acf']),
        'pit': pd.read_csv(required['pit']),
        'coverage': pd.read_csv(required['coverage']),
        'csv': required.copy(),
        'description': 'Loaded from cached CSV files.',
    }

    optional = {
        'diagnostic_series': base / 'diagnostic_series.csv',
        'jarque_bera': base / 'jarque_bera.csv',
        'series_summary': base / 'series_summary.csv',
        'series_values': base / 'series_values.csv',
    }

    for key, path in optional.items():
        if path.exists():
            result[key] = pd.read_csv(path)
            result['csv'][key] = path

    # Report-only additions. These never call model.fit().
    if 'series_summary' not in result:
        result['series_summary'] = make_series_summary(model_id, data)
        optional['series_summary'].parent.mkdir(parents=True, exist_ok=True)
        result['series_summary'].to_csv(optional['series_summary'], index=False)
        result['csv']['series_summary'] = optional['series_summary']

    if 'series_values' not in result:
        result['series_values'] = make_series_values(model_id, data)
        optional['series_values'].parent.mkdir(parents=True, exist_ok=True)
        result['series_values'].to_csv(optional['series_values'], index=False)
        result['csv']['series_values'] = optional['series_values']

    if 'diagnostic_series' not in result:
        theta = load_theta_csv(required['parameters'], model_id=model_id)
        result['diagnostic_series'] = make_is_diagnostic_series(model, theta, model_id, data)
        result['diagnostic_series'].to_csv(optional['diagnostic_series'], index=False)
        result['csv']['diagnostic_series'] = optional['diagnostic_series']

    if 'jarque_bera' not in result:
        result['jarque_bera'] = make_jb_from_diagnostic_series(model_id, result['diagnostic_series'])
        result['jarque_bera'].to_csv(optional['jarque_bera'], index=False)
        result['csv']['jarque_bera'] = optional['jarque_bera']

    return result


def fit_or_load_then_cache(seasonal: str, loc: str, data: dict, model_type: str):
    model_id = safe_model_id(seasonal, loc, model_type)
    model = build_zagas_model(model_type, seasonal=seasonal)

    if not FORCE_REFIT:
        cached = cached_result(model, model_id, data)
        if cached is not None:
            print(f'Loaded cached diagnostics: {model_id}')
            return cached

    output_dir = cache_output_dir_for_seasonal(seasonal)
    param_csv = output_dir / model_id / 'estimated_parameters.csv'
    if param_csv.exists() and not FORCE_REFIT:
        theta = load_theta_csv(param_csv, model_id=model_id)
        fit = {
            'theta': theta,
            'loglik': model.loglik(theta, data['y_train']),
            'success': True,
            'result': None,
        }
        print(f'Loaded theta, recomputing missing diagnostics: {model_id}')
    else:
        if seasonal == 'daily' and not ALLOW_DAILY_REFIT:
            raise RuntimeError(f'Missing daily cache for {model_id}. Daily refit is disabled by ALLOW_DAILY_REFIT=False.')
        print(f'Fitting {model_id}  n_train={len(data["y_train"]):,}  n_test={len(data["y_test"]):,}')
        if seasonal == 'monthly' and USE_MONTHLY_BFGS_TRIAL:
            print(f'  monthly optimizer: method={MONTHLY_FIT_METHOD}, use_bounds={MONTHLY_USE_BOUNDS}')
            fit = model.fit(data['y_train'], verbose=False, method=MONTHLY_FIT_METHOD, use_bounds=MONTHLY_USE_BOUNDS)
        else:
            fit = model.fit(data['y_train'], verbose=False)
        print(f'  loglik={fit["loglik"]:.3f} success={fit["success"]}')
        
        res = fit.get("result", None)

        if res is not None:
            print("  message:", getattr(res, "message", None))
            print("  status:", getattr(res, "status", None))
            print("  nit:", getattr(res, "nit", None))
            print("  fun:", getattr(res, "fun", None))

            jac = getattr(res, "jac", None)
            if jac is not None:
                print("  max|grad|:", np.max(np.abs(jac)))

    result = evaluate_and_cache_model(
        model=model,
        fit=fit,
        y_train=data['y_train'],
        y_test=data['y_test'],
        model_id=model_id,
        output_dir=output_dir,
        n_draws=N_DRAWS,
        seed=SEED,
    )
    result['description'] = f'{seasonal} {loc}: {model_type}'
    return result


In [6]:
all_results = []

for seasonal, dataset in [('daily', daily_data), ('monthly', monthly_data)]:
    for loc, data in dataset.items():
        for model_type in MODEL_TYPES:
            all_results.append(fit_or_load_then_cache(seasonal, loc, data, model_type))

print(f'Collected {len(all_results)} model result objects')

Fitting daily_belo_horizonte_phi  n_train=3,652  n_test=731


c:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\python attempt\FurtherTopics\models\za_gas_model.py:500: OptimizeWarning: Unknown solver options: ftol
  result = minimize(
c:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\python attempt\FurtherTopics\distributions\gb2_log_link.py:46: RuntimeWarning: overflow encountered in exp
  sigma = np.exp(phi)
c:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\python attempt\FurtherTopics\distributions\gb2_log_link.py:51: RuntimeWarning: divide by zero encountered in log
  + (a * p - 1) * np.log(y / sigma)
c:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\python attempt\FurtherTopics\distributions\gb2_log_link.py:49: RuntimeWarning: invalid value encountered in scalar add
  np.log(p)
c:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\python attempt\FurtherTopics\distributions\gb2_log_link.py:47: RuntimeWarning: divide by zero encountered in scalar divide
  z = (y / sigma) ** p
c:\Users\ilang\O

  loglik=-5086.473 success=False
  message: Desired error not necessarily achieved due to precision loss.
  status: 2
  nit: 145
  fun: 5086.4725151078055
  max|grad|: 3.0604248046875
Fitting daily_belo_horizonte_phi_xi  n_train=3,652  n_test=731


c:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\python attempt\FurtherTopics\models\za_gas_model.py:500: OptimizeWarning: Unknown solver options: ftol
  result = minimize(
c:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\python attempt\FurtherTopics\distributions\gb2_log_link.py:52: RuntimeWarning: divide by zero encountered in log
  - np.log(beta_fn(a, b))
c:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\python attempt\FurtherTopics\distributions\gb2_log_link.py:46: RuntimeWarning: overflow encountered in exp
  sigma = np.exp(phi)
c:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\python attempt\FurtherTopics\distributions\gb2_log_link.py:51: RuntimeWarning: divide by zero encountered in log
  + (a * p - 1) * np.log(y / sigma)
c:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\python attempt\FurtherTopics\distributions\gb2_log_link.py:49: RuntimeWarning: invalid value encountered in scalar add
  np.log(p)
c:\Users\ilang\OneDrive

  loglik=-5084.456 success=False
  message: Desired error not necessarily achieved due to precision loss.
  status: 2
  nit: 139
  fun: 5084.456111588879
  max|grad|: 189.7601318359375
Collected 2 model result objects


## Comparison tables

In [7]:
metrics = pd.concat([r['metrics_frame'] for r in all_results], ignore_index=True)
coverage = pd.concat([r['coverage'] for r in all_results], ignore_index=True)
acf = pd.concat([r['acf'] for r in all_results], ignore_index=True)
pit = pd.concat([r['pit'] for r in all_results], ignore_index=True)
params = pd.concat([r['parameters'] for r in all_results], ignore_index=True)
jarque_bera_tables = pd.concat([r['jarque_bera'] for r in all_results if 'jarque_bera' in r], ignore_index=True)
series_summary = pd.concat([r['series_summary'] for r in all_results if 'series_summary' in r], ignore_index=True)

def readable(df):
    out = df.copy()
    if 'model_id' in out.columns:
        out.insert(0, 'Model', out['model_id'].map(display_model_label))
    return out

display(readable(metrics))
display(readable(coverage))
display(readable(acf).pivot_table(index=['Model', 'sample'], columns='lag', values='acf'))
display(readable(pit))
display(readable(params).head(20))
display(readable(jarque_bera_tables))
display(series_summary.drop_duplicates(subset=['series_id', 'sample']))

,Model,model_id,sample,loglik,aic,bic,rmse,mad,crps
0,Daily $\phi$-only,daily_belo_horizonte_phi,IS,-5086.470241,10204.940482,10302.499303,66.517190,46.659242,2.743713
1,Daily $\phi$-only,daily_belo_horizonte_phi,OOS,-1131.906503,2295.813006,2369.323621,64.080321,46.138860,2.703975
2,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,IS,-5084.456112,10214.912223,10355.153028,11.218092,5.568192,2.724289
3,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,OOS,-1134.150700,2314.301400,2419.972910,10.509041,5.600799,2.706262


,Model,model_id,sample,coverage,alpha,violations,expected_violations,violation_rate,LR_uc,pvalue_uc,LR_ind,LR_cc,pvalue_cc,reject_uc_5pct,reject_cc_5pct
0,Daily $\phi$-only,daily_belo_horizonte_phi,IS,95%,0.05,140,164.30,0.042605,3.974219,0.046202,1.473551,5.447770,0.065619,True,False
1,Daily $\phi$-only,daily_belo_horizonte_phi,OOS,95%,0.05,27,36.55,0.036936,2.877139,0.089846,0.854952,3.732092,0.154734,False,False
2,Daily $\phi$-only,daily_belo_horizonte_phi,Full,95%,0.05,167,200.85,0.041573,6.354813,0.011706,2.219564,8.574377,0.013744,True,True
3,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,IS,95%,0.05,140,164.30,0.042605,3.974219,0.046202,3.782429,7.756647,0.020685,True,True
4,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,OOS,95%,0.05,28,36.55,0.038304,2.282144,0.130871,0.005630,2.287774,0.318578,False,False
5,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,Full,95%,0.05,168,200.85,0.041822,5.974258,0.014516,3.213337,9.187595,0.010114,True,True


lag                            1         2         3         365       366
Model             sample                                                  
Daily $\phi$-only IS      0.091716  0.111097  0.060993  0.052744 -0.000163
                  OOS     0.154325  0.109026  0.086171  0.001180 -0.012436
Daily $\phi,\xi$  IS      0.081942  0.105315  0.055508  0.020011  0.017296
                  OOS     0.129308  0.117634  0.088605  0.018411  0.002677

,Model,n,ks_stat,pvalue,reject_5pct,model_id,sample
0,Daily $\phi$-only,3286,0.019552,0.159970,False,daily_belo_horizonte_phi,IS
1,Daily $\phi$-only,731,0.030114,0.511527,False,daily_belo_horizonte_phi,OOS
2,Daily $\phi$-only,4017,0.016302,0.233521,False,daily_belo_horizonte_phi,Full
3,"Daily $\phi,\xi$",3286,0.025096,0.031340,True,daily_belo_horizonte_phi_xi,IS
4,"Daily $\phi,\xi$",731,0.029794,0.525374,False,daily_belo_horizonte_phi_xi,OOS
5,"Daily $\phi,\xi$",4017,0.018560,0.124046,False,daily_belo_horizonte_phi_xi,Full


,Model,model_id,sample,seasonal,model_tv_params,parameter_order,parameter,block,estimate,std_error,ci_lower_95,ci_upper_95,loglik,success
0,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,0,omega_phi,gas,2.109644e-01,0.021635,0.168560,0.253369,-5086.472515,False
1,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,1,f0_phi,gas,1.078085e+01,0.251673,10.287580,11.274120,-5086.472515,False
2,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,2,A_phi_1,gas,7.776805e-02,0.025354,0.028076,0.127461,-5086.472515,False
3,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,3,A_phi_365,gas,9.966699e-02,0.040439,0.020408,0.178926,-5086.472515,False
4,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,4,A_phi_366,gas,-5.678814e-02,0.042006,-0.139119,0.025542,-5086.472515,False
5,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,5,B_phi_1,gas,9.266299e-01,0.035245,0.857551,0.995708,-5086.472515,False
6,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,6,B_phi_365,gas,1.325619e-07,0.027996,-0.054872,0.054872,-5086.472515,False
7,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,7,B_phi_366,gas,5.352078e-02,0.048001,-0.040559,0.147600,-5086.472515,False
8,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,8,xi,positive_static,-5.161364e-02,0.002103,-0.055735,-0.047493,-5086.472515,False
9,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,9,gamma,positive_static,1.980916e-01,0.022166,0.154648,0.241536,-5086.472515,False


,Model,model_id,sample,n,skewness,kurtosis,JB_stat,pvalue,reject_5pct
0,Daily $\phi$-only,daily_belo_horizonte_phi,IS,3286,-0.063520,2.989232,2.225574,0.328642,False
1,Daily $\phi$-only,daily_belo_horizonte_phi,OOS,731,0.044121,2.927544,0.397070,0.819931,False
2,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,IS,3286,-0.094465,3.073292,5.622602,0.060127,False
3,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,OOS,731,0.029248,3.001988,0.104339,0.949168,False


,series_id,model_id,sample,n,min,max,mean,std,prop_zero,q01,q05,q25,q50,q75,q90,q95,q99
0,daily_belo_horizonte,daily_belo_horizonte_phi,IS,3652,0.0,171.8,4.007694,11.629553,0.723165,0.0,0.0,0.0,0.0,0.5,13.9,25.545,53.349
1,daily_belo_horizonte,daily_belo_horizonte_phi,OOS,731,0.0,92.8,4.153352,10.836550,0.705882,0.0,0.0,0.0,0.0,0.6,15.0,29.050,50.280
2,daily_belo_horizonte,daily_belo_horizonte_phi,Full,4383,0.0,171.8,4.031987,11.500029,0.720283,0.0,0.0,0.0,0.0,0.5,14.0,26.470,52.326


## OOS forecast cache

Each model has an `oos_forecasts.csv` with expected value, requested quantiles, simulated min/max, and observed value.

In [8]:
for r in all_results:
    print(r['model_id'], r['csv']['oos_forecasts'])
    display(r['oos_forecasts'].head())

daily_belo_horizonte_phi artifacts\cache_daily_lags_1_365_366_050626\daily_belo_horizonte_phi\oos_forecasts.csv


,model_id,sample,t_index,expected_value,q50,q75,q90,q95,q97,q99,sim_min,sim_max,observed
0,daily_belo_horizonte_phi,OOS,0,172.488278,6.845316,17.979183,35.136743,49.376314,60.417582,85.468221,0.0,139.047593,9.9
1,daily_belo_horizonte_phi,OOS,1,81.407429,0.000000,5.385153,21.091861,35.049137,46.113142,71.639010,0.0,151.798402,8.9
2,daily_belo_horizonte_phi,OOS,2,142.262639,4.594815,14.765683,30.791177,44.190234,54.609635,78.304901,0.0,175.198803,21.6
3,daily_belo_horizonte_phi,OOS,3,141.736347,2.811062,14.181213,32.854500,48.648580,60.982214,89.124664,0.0,182.143436,35.4
4,daily_belo_horizonte_phi,OOS,4,195.340618,7.663415,20.366765,39.969766,56.246786,68.870301,97.515322,0.0,167.087098,23.1


daily_belo_horizonte_phi_xi artifacts\cache_daily_lags_1_365_366_050626\daily_belo_horizonte_phi_xi\oos_forecasts.csv


,model_id,sample,t_index,expected_value,q50,q75,q90,q95,q97,q99,sim_min,sim_max,observed
0,daily_belo_horizonte_phi_xi,OOS,0,12.087335,6.317616,18.448735,37.009555,51.968854,63.300605,88.265168,0.0,139.013899,9.9
1,daily_belo_horizonte_phi_xi,OOS,1,6.043276,0.000000,6.424632,23.407970,37.276861,47.759883,70.754979,0.0,136.540611,8.9
2,daily_belo_horizonte_phi_xi,OOS,2,9.502175,4.339180,14.648140,30.244549,42.704996,52.103016,72.725533,0.0,150.152108,21.6
3,daily_belo_horizonte_phi_xi,OOS,3,10.009024,3.372759,15.575583,33.488940,47.554824,58.080709,81.019194,0.0,150.358008,35.4
4,daily_belo_horizonte_phi_xi,OOS,4,13.700049,8.028477,21.338025,40.712258,56.004161,67.486660,92.596341,0.0,149.787933,23.1


## Render PDF and LaTeX report

Produces a structured multi-section PDF (and a `.tex` sidecar) with:

1. **Cover page** — abstract and scope note (Belo Horizonte pilot only).
2. **Model descriptions** — ZA-GAS framework, lag sets, phi-only vs phi-xi, parameter counts.
3. **Per-model diagnostics** (one section per model):
   - In-sample PIT histogram
   - In-sample quantile residual ACF (short-range + seasonal lags)
   - KS uniform-fit test table
   - Estimated parameter table with 95 % CIs
4. **Comparison tables** (daily and monthly separately):
   - Performance metrics (IS and OOS) — best value per metric highlighted in green
   - Kupiec / Christoffersen 95 % coverage tests — rejected cells highlighted in red
5. **Conclusion** — summary, next steps, references.

All pages are numbered. The LaTeX sidecar mirrors the structure with `\cellcolor` highlights.

Report regeneration does not call the optimizer by itself; it only uses the `all_results` list created above. To rebuild the same report after fitting/loading, rerun this cell. To make a monthly-only report, use `RUN_SCOPE = 'monthly'` before rebuilding `all_results`; to make the full daily-plus-monthly report, use `RUN_SCOPE = 'both'`.

In [9]:
report_paths = render_model_report(
    all_results,
    pdf_path=REPORT_DIR / f'zagas_daily_comparison_{DAILY_LAG_TAG}.pdf',
    tex_path=REPORT_DIR / f'zagas_daily_comparison_{DAILY_LAG_TAG}.tex',
    title='ZA-GAS Daily Lag-Robustness Test - lags 1,365, 366 — Belo Horizonte',
)
report_paths

{'pdf': WindowsPath('artifacts/reports_daily_lags_1_365_366_050626/zagas_daily_comparison_lags_1_365_366_050626.pdf'),
 'tex': WindowsPath('artifacts/reports_daily_lags_1_365_366_050626/zagas_daily_comparison_lags_1_365_366_050626.tex')}